In [0]:
# Exploring the raw FOCUS billing data before building the bronze table

import pyspark.sql.functions as F
from pyspark.sql.types import *

volume_path = "/Volumes/finops_catalog/focus_billing_schema/raw_ingestion_landing_volume"

print("Reading raw CSV data...")
raw_df = (spark.read.format("csv")
          .option("header", "true")
          .option("inferSchema", "true")
          .load(volume_path))

print(f"Loaded {raw_df.count():,} records\n")

# Check the schema
print("Schema:")
raw_df.printSchema()

# Count columns by data type
schema_summary = {}
for field in raw_df.schema.fields:
    dtype = str(field.dataType)
    schema_summary[dtype] = schema_summary.get(dtype, 0) + 1

print("\nData type distribution:")
for dtype, count in sorted(schema_summary.items()):
    print(f"  {dtype}: {count} columns")

print(f"\nTotal: {len(raw_df.columns)} columns, {raw_df.count():,} rows")

In [0]:
# Check data quality

print("Checking for null values...")
null_counts = raw_df.select([F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) 
                             for c in raw_df.columns])

null_summary = []
for col_name in raw_df.columns:
    null_count = null_counts.collect()[0][col_name]
    null_pct = (null_count / raw_df.count()) * 100
    if null_pct > 0:
        null_summary.append((col_name, null_count, null_pct))

# Sort by null percentage descending
null_summary.sort(key=lambda x: x[2], reverse=True)

if null_summary:
    print(f"\n  Found {len(null_summary)} columns with NULL values:\n")
    for col_name, null_count, null_pct in null_summary[:10]:  # Top 10
        print(f"  {col_name:30s} | {null_count:>7,} nulls ({null_pct:>5.1f}%)")
    if len(null_summary) > 10:
        print(f"  ... and {len(null_summary) - 10} more columns with nulls")
else:
    print("  ✅ No NULL values found!")

# Check for 'NULL' string literals
print("\nChecking for 'NULL' strings...")
string_cols = [field.name for field in raw_df.schema.fields if str(field.dataType) == "StringType"]

null_string_counts = {}
for col in string_cols:
    count = raw_df.filter(F.col(col) == "NULL").count()
    if count > 0:
        null_string_counts[col] = count

# Check for duplicates
print("\nChecking for duplicates...")
total_ids = raw_df.select("Id").count()
unique_ids = raw_df.select("Id").distinct().count()
duplicates = total_ids - unique_ids

if duplicates > 0:
    print(f"Found {duplicates:,} duplicate records ({(duplicates/total_ids)*100:.2f}%)")
else:
    print(f"All {unique_ids:,} records are unique")

In [0]:
# Look at cost distribution

print("Cost summary:")
cost_stats = raw_df.select(
    F.sum("BilledCost").alias("total_billed"),
    F.sum("EffectiveCost").alias("total_effective"),
    F.avg("BilledCost").alias("avg_billed"),
    F.min("BilledCost").alias("min_billed"),
    F.max("BilledCost").alias("max_billed"),
    F.count(F.when(F.col("BilledCost") == 0, 1)).alias("zero_cost_count"),
    F.count(F.when(F.col("BilledCost") < 0, 1)).alias("negative_cost_count")
).collect()[0]

print(f"  Total Billed: ${cost_stats['total_billed']:,.2f}")
print(f"  Total Effective: ${cost_stats['total_effective']:,.2f}")
print(f"  Average: ${cost_stats['avg_billed']:,.4f}")
print(f"  Range: ${cost_stats['min_billed']:,.2f} to ${cost_stats['max_billed']:,.2f}")
print(f"\n  Zero-cost records: {cost_stats['zero_cost_count']:,} ({(cost_stats['zero_cost_count']/raw_df.count())*100:.1f}%)")
print(f"  Negative costs: {cost_stats['negative_cost_count']:,}")



In [0]:
# Check what services are being used

print("Top 10 services by record count:")
top_services = raw_df.groupBy("ServiceName") \
    .agg(
        F.count("*").alias("record_count"),
        F.sum("BilledCost").alias("total_cost")
    ) \
    .orderBy(F.desc("record_count")) \
    .limit(10) \
    .collect()

for i, row in enumerate(top_services, 1):
    service = row['ServiceName'] or "(NULL)"
    count = row['record_count']
    cost = row['total_cost'] or 0
    pct = (count / raw_df.count()) * 100
    print(f"  {i:2d}. {service[:40]:40s} | {count:>6,} records ({pct:>5.1f}%) | ${cost:>12,.2f}")

# Top services by cost
print("\nTop 10 services by total cost:")
top_cost_services = raw_df.groupBy("ServiceName") \
    .agg(
        F.sum("BilledCost").alias("total_cost"),
        F.count("*").alias("record_count")
    ) \
    .orderBy(F.desc("total_cost")) \
    .limit(10) \
    .collect()

for i, row in enumerate(top_cost_services, 1):
    service = row['ServiceName'] or "(NULL)"
    cost = row['total_cost'] or 0
    count = row['record_count']
    print(f"  {i:2d}. {service[:40]:40s} | ${cost:>12,.2f} | {count:>6,} records")

# Count unique values
print("\nUnique counts:")
cardinality = raw_df.select(
    F.countDistinct("ServiceName").alias("unique_services"),
    F.countDistinct("ResourceId").alias("unique_resources"),
    F.countDistinct("RegionId").alias("unique_regions"),
).collect()[0]

print(f"  Services: {cardinality['unique_services']:,}")
print(f"  Resources: {cardinality['unique_resources']:,}")
print(f"  Regions: {cardinality['unique_regions']:,}")


In [0]:
# Summary of findings

print("\nKey findings:")
findings = []

# Check for NULL strings
if null_string_counts:
    findings.append(f"Found 'NULL' string literals in {len(null_string_counts)} columns - will need cleansing")

# Check zero cost records
zero_cost_pct = (cost_stats['zero_cost_count'] / raw_df.count()) * 100
if zero_cost_pct > 10:
    findings.append(f"{zero_cost_pct:.1f}% of records have zero cost - investigate if expected")

# Check data type issues
cost_string_cols = [c for c in ["ContractedCost", "ListCost"] if c in string_cols]
if cost_string_cols:
    findings.append(f"Cost columns stored as strings: {', '.join(cost_string_cols)} - will need casting")

# Check null resources
resource_null_pct = (null_counts.collect()[0]['ResourceId'] / raw_df.count()) * 100
if resource_null_pct > 20:
    findings.append(f"{resource_null_pct:.1f}% of records missing ResourceId - may impact cost allocation")

# Check duplicates
if duplicates > 0:
    findings.append(f"{duplicates:,} duplicate records detected - will need deduplication")

if findings:
    print("\nIssues to address:")
    for i, finding in enumerate(findings, 1):
        print(f"  {i}. {finding}")
else:
    print("No major data quality issues found!")

print("\nData looks good - ready to create the bronze table.")